# Escape velocity

*astro · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/escape_velocity)

A batch of rockets leaves a planet with speeds around the escape speed $v_\mathrm{esc} = \sqrt{2GM/R}$. The sign of the energy decides each rocket's fate: bound rockets ($E < 0$) fall back, unbound ones ($E \ge 0$) never return.

**Vector form**

$$\ddot{\mathbf r} = -\frac{GM}{r^3}\,\mathbf r,\qquad E = \tfrac12 v^2 - \frac{GM}{r},\qquad v_\mathrm{esc} = \sqrt{\frac{2GM}{R}}$$

**Index (tensor) form**

$$\ddot x_{bi} = -GM\,(x_{bj}x_{bj})^{-3/2}\,x_{bi},\qquad E_b = \tfrac12\,v_{bi}v_{bi} - GM\,(x_{bj}x_{bj})^{-1/2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `y[:, 0, :]` | (B, 2) | position of rocket $b$ from the planet's centre |
| $v_{bi}$ | `y[:, 1, :]` | (B, 2) | velocity of rocket $b$ |
| $x_{bj}x_{bj}$ | `einsum('bj,bj->b', x, x)` | (B,) | repeated $j$ ⇒ summed: squared distance $r_b^2$ |
| $\ddot x_{bi}$ | `acceleration(x, GM)` | (B, 2) | gravity of the planet |
| $E_b$ | `energy(y, GM)` | (B,) | energy per unit mass: negative ⇒ bound |
| $v_\mathrm{esc}$ | `escape_speed(GM, R)` | () | speed with $E = 0$ at the surface |
| $R_s$ | `dark_star_radius(GM, c)` | () | $2GM/c^2$: radius at which $v_\mathrm{esc} = c$ |
| $r_\mathrm{max}$ | `max_radius(v, GM, R)` | (B,) | $R/(1 - v^2/v_\mathrm{esc}^2)$, highest point of a vertical launch |

Units: planet radius $R = 1$ and $GM = 1$, so $v_\mathrm{esc} = \sqrt 2$ (for the Earth, 11.2 km/s). The rockets coast with the engine off, there is no air, and the planet does not rotate. Just below $v_\mathrm{esc}$ the highest point shoots up as $1/(1 - v^2/v_\mathrm{esc}^2)$, and the trip takes longer and longer. Exactly at $v_\mathrm{esc}$ the rocket slows down forever but never stops. Tilt the launches to see that the escape speed does not depend on direction (as long as the path does not hit the planet): only the energy matters. Dark stars: squeeze the planet until $v_\mathrm{esc}$ reaches the speed of light $c$ and you get $R = 2GM/c^2$. Michell (1783) and Laplace argued that such a star would be invisible. General relativity gives exactly the same radius for the horizon of a black hole (the Schwarzschild radius, 3 km for the Sun), although the Newtonian reasoning behind it is not the real story.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/escape_velocity/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    GM, R = tensors.GM, tensors.R
    v_esc = equation.escape_speed(GM, R)
    fractions = tensors.speed_fractions(params["n_rockets"], params["f_min"], params["f_max"])
    y0 = tensors.initial_state(fractions, v_esc, params["tilt"], dev)

    times, traj = solve.solve(y0, GM, R, params["t_max"], params["dt"])
    E = equation.energy(y0, GM)  # (B,) fixed by the launch
    gone = solve.escaped(traj, R)
    r_top = solve.highest_point(traj)
    r_max = equation.max_radius(fractions * v_esc, GM, R) if params["tilt"] == 0 else None

    print(f"v_esc = √(2GM/R) = {v_esc:.4f}")
    for b, f in enumerate(fractions.tolist()):
        if abs(E[b]) < 1e-5:
            fate = "E = 0: slows down forever, never returns"
        else:
            fate = "escapes" if gone[b] else ("falls back" if E[b] < 0 else "still climbing")
        print(f"v = {f:.2f} v_esc   E = {E[b]:+.3f}   highest r = {r_top[b]:6.2f} R   {fate}")
    GM_sun, c = 1.327e20, 2.998e8  # m³/s², m/s
    print(f"dark star: v_esc = c when R = 2GM/c², e.g. {equation.dark_star_radius(GM_sun, c) / 1e3:.2f} km for the Sun")
    return plot.animate(times, traj, fractions, r_max, R, out_dir / "escape_velocity.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/escape_velocity/equation.py
r"""Physics → tensor translation.

A rocket coasting (engine off) above a planet of mass M only feels the planet's gravity:

    \ddot{\mathbf r} = -\frac{GM}{r^3}\,\mathbf r

For a batch of rockets b and components i ∈ {x, y} (repeated j ⇒ summed):

    \ddot x_{bi} = -GM\,(x_{bj} x_{bj})^{-3/2}\,x_{bi}

The energy per unit mass is conserved,

    E_b = ½ v_{bi} v_{bi} - \frac{GM}{\sqrt{x_{bj} x_{bj}}},

and decides everything: E < 0 → the rocket is bound and falls back, E ≥ 0 → it escapes to infinity.
Setting E = 0 at the surface r = R gives the escape speed v_esc = \sqrt{2GM/R}. A rocket fired
straight up with v < v_esc stops at the height where all its kinetic energy is used up:

    ½ v^2 - GM/R = -GM/r_max   ⇒   r_max = R / (1 - v^2 / v_esc^2)

Michell (1783) and Laplace (1796) asked how small a star must be for v_esc to reach the speed of light c.
Setting \sqrt{2GM/R} = c gives the "dark star" radius R = 2GM/c^2 — exactly the Schwarzschild radius of a
black hole in general relativity (a lucky coincidence of the Newtonian argument).
"""

import math

import torch


def acceleration(x: torch.Tensor, GM: float) -> torch.Tensor:
    r"""\ddot x_{bi} = -GM (x_{bj} x_{bj})^{-3/2} x_{bi}.   x: (B, 2) → (B, 2)"""
    r2 = torch.einsum("bj,bj->b", x, x)  # (B,) squared distance from the planet's centre
    return -GM * torch.einsum("b,bi->bi", r2**-1.5, x)


def rhs(y: torch.Tensor, GM: float) -> torch.Tensor:
    """dy/dt for the state y of shape (B, 2, 2): y[:, 0] = position, y[:, 1] = velocity."""
    return torch.stack([y[:, 1], acceleration(y[:, 0], GM)], dim=1)


def energy(y: torch.Tensor, GM: float) -> torch.Tensor:
    r"""E_b = ½ v_{bi} v_{bi} - GM / r_b.   y: (..., 2, 2) → (...)"""
    x, v = y[..., 0, :], y[..., 1, :]
    return 0.5 * torch.einsum("...i,...i->...", v, v) - GM / torch.einsum("...i,...i->...", x, x).sqrt()


def escape_speed(GM: float, R: float) -> float:
    r"""v_esc = \sqrt{2GM/R}: the launch speed with exactly zero total energy."""
    return math.sqrt(2 * GM / R)


def max_radius(v: torch.Tensor, GM: float, R: float) -> torch.Tensor:
    """Highest point of a vertical launch with v < v_esc, from energy conservation."""
    return R / (1 - v**2 / escape_speed(GM, R) ** 2)


def dark_star_radius(GM: float, c: float) -> float:
    """Radius at which the escape speed equals c: R = 2GM/c² (the Schwarzschild radius)."""
    return 2 * GM / c**2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/escape_velocity/tensors.py
"""Build the concrete tensors: launch speeds, launch sites and the batched initial state.

Units: the planet's radius R = 1 and GM = 1, so the escape speed is √2 and one time unit is
√(R³/GM) (for the Earth about 13 minutes).
"""

import math

import torch

GM = 1.0
R = 1.0


def speed_fractions(n: int, f_min: float, f_max: float) -> torch.Tensor:
    """Launch speeds as fractions v / v_esc, (B,)."""
    return torch.linspace(f_min, f_max, n) if n > 1 else torch.tensor([f_min])


def initial_state(fractions: torch.Tensor, v_esc: float, tilt_deg: float, device: torch.device) -> torch.Tensor:
    """y0 of shape (B, 2, 2): rockets spread over the upper half of the planet, each launched outward.

    Rocket b sits on the surface at polar angle φ_b and is fired at speed f_b v_esc, tilted by tilt_deg
    from the local vertical (0 = straight up).
    """
    B = len(fractions)
    phi = torch.linspace(math.radians(160), math.radians(20), B) if B > 1 else torch.tensor([math.pi / 2])
    up = torch.stack([torch.cos(phi), torch.sin(phi)], dim=-1)  # (B, 2) local vertical
    side = torch.stack([torch.sin(phi), -torch.cos(phi)], dim=-1)  # (B, 2) local horizontal
    tilt = math.radians(tilt_deg)
    direction = math.cos(tilt) * up + math.sin(tilt) * side
    position = R * up
    velocity = (fractions * v_esc)[:, None] * direction
    return torch.stack([position, velocity], dim=1).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/escape_velocity/solve.py
"""Integrate all rockets at once with RK4; a rocket that falls back is frozen where it lands."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(y0: torch.Tensor, GM: float, R: float, t_max: float, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2, 2))."""
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, GM), y0, dt, n_steps)

    r = traj[:, :, 0, :].norm(dim=-1)  # (T, B) distance from the centre
    below = r < R * (1 - 1e-4)  # small margin: at launch r = R up to round-off
    landed = torch.cummax(below.int(), dim=0).values.bool()  # stays True after touchdown
    step = torch.where(landed.any(0), landed.int().argmax(0), len(r) - 1)  # (B,)
    rest = traj[step, torch.arange(traj.shape[1])].clone()  # (B, 2, 2)
    rest[:, 0] = R * rest[:, 0] / rest[:, 0].norm(dim=-1, keepdim=True)  # back on the surface
    rest[:, 1] = 0.0
    traj = torch.where(landed[:, :, None, None], rest, traj)

    times = torch.arange(n_steps + 1, device=y0.device, dtype=y0.dtype) * dt
    return times, traj


def escaped(traj: torch.Tensor, R: float) -> torch.Tensor:
    """True for rockets still in flight at the end and moving away from the planet, (B,)."""
    x, v = traj[-1, :, 0], traj[-1, :, 1]
    return (x.norm(dim=-1) > 1.01 * R) & (torch.einsum("bi,bi->b", x, v) > 0)


def highest_point(traj: torch.Tensor) -> torch.Tensor:
    """Largest distance from the centre reached by each rocket, (B,)."""
    return traj[:, :, 0, :].norm(dim=-1).max(dim=0).values


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/escape_velocity/plot.py
"""Animate the launches around the planet next to the distance r(t) of every rocket."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    times: torch.Tensor,
    traj: torch.Tensor,
    fractions: torch.Tensor,
    r_max: torch.Tensor | None,
    R: float,
    path: Path,
    view: float = 7.0,
) -> Path:
    """r_max: predicted highest points (B,) for vertical launches, or None when the launches are tilted."""
    t = times.cpu().numpy()
    pos = traj[:, :, 0, :].cpu().numpy() / R  # (T, B, 2) in planet radii
    r = np.linalg.norm(pos, axis=-1)  # (T, B)
    f = fractions.cpu().numpy()
    # blue shades = bound (v < v_esc), red shades = unbound (v ≥ v_esc)
    colors = [
        plt.cm.Blues(np.interp(fb, [f.min(), 1], [0.4, 0.95])) if fb < 1 - 1e-6 else plt.cm.Reds(0.5 + 0.4 * (fb - 1))
        for fb in f
    ]
    B = len(f)

    fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1.15, 1]})
    ax.set_xlim(-view, view)
    ax.set_ylim(-0.3 * view, view)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x / R")
    ax.set_ylabel("y / R")
    ax.add_patch(plt.Circle((0, 0), 1.0, color="#93c5fd", ec="#1e40af", lw=1.5, zorder=3))
    lines = [ax.plot([], [], lw=1.3, color=colors[b])[0] for b in range(B)]
    dots = [ax.plot([], [], "o", ms=5, color=colors[b], zorder=4)[0] for b in range(B)]
    clock = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top", fontsize=9)
    dark_star = "dark star: v_esc = c  ⇔  R = 2GM/c²"
    ax.text(0.97, 0.03, dark_star, transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="#374151")
    ax.set_title("blue: bound (E < 0)   red: escapes (E ≥ 0)", fontsize=9)

    top = 12.0
    ax_r.set_xlim(t[0], t[-1])
    ax_r.set_ylim(0, top)
    ax_r.set_xlabel("t  (units of √(R³/GM))")
    ax_r.set_ylabel("r / R")
    ax_r.axhline(1.0, color="#1e40af", lw=1)
    curves = [ax_r.plot([], [], lw=1.4, color=colors[b], label=f"{f[b]:.2f}")[0] for b in range(B)]
    if r_max is not None:
        for b, rm in enumerate(r_max.cpu().numpy() / R):
            if np.isfinite(rm) and 0 < rm < top:
                ax_r.axhline(rm, color=colors[b], lw=0.8, ls="--")
        ax_r.set_title("dashed: r_max from energy conservation", fontsize=9)
    ax_r.legend(fontsize=7, ncol=2, loc="upper right", title="v / v_esc", title_fontsize=7)

    def update(step: int):
        for b in range(B):
            lines[b].set_data(pos[: step + 1, b, 0], pos[: step + 1, b, 1])
            dots[b].set_data([pos[step, b, 0]], [pos[step, b, 1]])
            curves[b].set_data(t[: step + 1], r[: step + 1, b])
        clock.set_text(f"t = {t[step]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_rockets': 8,
    'f_min': 0.6,
    'f_max': 1.3,
    'tilt': 0.0,
    't_max': 30.0,
    'dt': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.escape_velocity")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.escape_velocity.main")
problem_dir = Path("problems/astro/escape_velocity")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
